# RAG Evaluation & Experimentation — Phase 2

Phase 1 built a *working* RAG pipeline on SEC filings. This notebook makes it a **measured**
one — the evaluation / guardrails / monitoring feedback loop that most RAG tutorials skip, and
the part that proves what you ship actually works.

```
        Phase 1 pipeline                          Phase 2 (this notebook)
01→02→03→04→05  ──────────────▶   gold Q/A set → retrieval metrics → LLM-judge → 1 experiment
                                   (recall@k / MRR)  (faithfulness)   (change one knob, measure)
```

### The stack (hybrid)
- **Generation:** local **Mistral-7B** (4-bit) — the pipeline we're evaluating.
- **Judge:** the **Claude API** (`claude-opus-4-8`) — a stronger model grades faithfulness &
  relevance. A 7B model grading its own output is a weak judge; this is more credible.

### Memory-safe ordering (you hit an OOM last time)
The **retrieval half runs first with no Mistral loaded** (embeddings + vector store only), so
even if the T4 OOMs later, you keep those results. Generation loads Mistral **last**, and there's
a `USE_LOCAL_LLM = False` switch that generates with Claude instead if the GPU won't cooperate.

> **Runtime:** Colab **T4 GPU**. Secrets (key icon): `HF_TOKEN` *and* `ANTHROPIC_API_KEY`.
> Accept the Mistral-7B-Instruct-v0.3 license on Hugging Face. Run top-to-bottom once.

## Setup

In [ ]:
!pip install -qU langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -qU qdrant-client sentence-transformers
!pip install -qU transformers accelerate bitsandbytes
!pip install -qU beautifulsoup4 lxml anthropic

In [ ]:
import os, re, time, json, textwrap, random
import requests
import numpy as np
from bs4 import BeautifulSoup
from google.colab import userdata

# Two secrets required in Colab (key icon, left panel)
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

import anthropic
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import CrossEncoder

SEC_USER_AGENT = "David Schaaf davidschaaf@berkeley.edu"   # edit to your name/email
JUDGE_MODEL = "claude-opus-4-8"
client = anthropic.Anthropic()                            # reads ANTHROPIC_API_KEY
print("Setup ready. Claude judge:", JUDGE_MODEL)

## Rebuild the Phase-1 pipeline *(provided — you built this in Phase 1)*

Fetch → clean → chunk (each chunk gets a stable `chunk_id` so we can check retrieval exactly) →
embed → index in Qdrant → build a reranked retriever. **No Mistral yet** — none of the retrieval
evaluation below needs a GPU LLM.

In [ ]:
# --- fetch + clean + chunk (with chunk_id) ---
COMPANIES = {"AAPL": 320193, "MSFT": 789019, "NVDA": 1045810}

def fetch_latest_10k_html(ticker, cik):
    headers = {"User-Agent": SEC_USER_AGENT}
    recent = requests.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json",
                          headers=headers).json()["filings"]["recent"]
    i = next(j for j, f in enumerate(recent["form"]) if f == "10-K")
    acc, doc = recent["accessionNumber"][i].replace("-", ""), recent["primaryDocument"][i]
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{doc}"
    html = requests.get(url, headers=headers).text
    time.sleep(0.5)
    return {"ticker": ticker, "filing_date": recent["filingDate"][i], "html": html}

def clean_filing_text(html):
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ")).strip()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
documents = []
for tk, cik in COMPANIES.items():
    print("Fetching", tk, "10-K ...")
    f = fetch_latest_10k_html(tk, cik)
    for chunk in splitter.split_text(clean_filing_text(f["html"])):
        documents.append(Document(page_content=chunk,
            metadata={"chunk_id": len(documents), "ticker": tk, "filing_date": f["filing_date"]}))
print("Total chunks:", len(documents))

In [ ]:
# --- embed + index + reranked retrieval ---
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
qdrant = QdrantVectorStore.from_documents(
    documents, embeddings, location=":memory:", collection_name="sec_eval")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_baseline(query, k=10):
    return qdrant.similarity_search(query, k=k)

def retrieve_reranked(query, fetch_k=10, top_k=10):
    cands = qdrant.similarity_search(query, k=fetch_k)
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return [d for _, d in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)][:top_k]

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['ticker']}] {d.page_content}" for d in docs)

print("Pipeline ready (no GPU LLM loaded yet).")

---
## 1 · Build a gold evaluation set *(synthetic)*

To measure retrieval you need ground truth: for a known chunk, a question that chunk answers.
We use Claude to generate them — for each sampled chunk, one question answerable **only** from
that chunk. The chunk's `chunk_id` is then the *correct* retrieval for that question. This is a
standard, automatable way to get a labeled eval set without hand-writing Q/A pairs.

In [ ]:
# --- PROVIDED: a tiny Claude helper (plain text answer) ---
def ask_claude(prompt, max_tokens=256):
    msg = client.messages.create(
        model=JUDGE_MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text.strip()

print("Claude helper ready.")

In [ ]:
# --- YOU WRITE: generate one question a given chunk uniquely answers ---
def gen_question(chunk_text):
    prompt = (
        "Below is an excerpt from an SEC 10-K filing. Write ONE specific question that is "
        "answered ONLY by this excerpt. Return just the question, nothing else.\n\n"
        f"Excerpt:\n{chunk_text}"
    )
    return ask_claude(prompt)

print(gen_question(documents[len(documents)//2].page_content))

In [ ]:
# --- PROVIDED: sample content-rich chunks, build the gold set ---
N_GOLD = 12
random.seed(7)
rich = [d for d in documents if len(d.page_content) > 600
        and sum(c.isalpha() for c in d.page_content) / len(d.page_content) > 0.7]
sampled = random.sample(rich, N_GOLD)

gold = []
for d in sampled:
    gold.append({"question": gen_question(d.page_content),
                 "source_id": d.metadata["chunk_id"], "ticker": d.metadata["ticker"]})
print(f"Built {len(gold)} gold Q/A pairs.")

In [ ]:
# --- INSTRUMENT: eyeball the gold set ---
for g in gold[:3]:
    src = documents[g["source_id"]].page_content[:120].replace("\n", " ")
    print(f"[{g['ticker']}] Q: {g['question']}")
    print(f"        source chunk {g['source_id']}: {src}...\n")

**🎤 Talk-track — Gold set.** *"You can't improve what you don't measure, and RAG's failure
mode is silent — a confident answer over the wrong context. I bootstrap a labeled eval set with an
LLM: generate a question from a known chunk, and now I know exactly which chunk retrieval should
return. It's cheap, repeatable, and gives me ground truth to score against."*

---
## 2 · Retrieval metrics — recall@k & MRR  *(no GPU needed)*

Retrieval is the ceiling on the whole system: if the right chunk never reaches the prompt, no
LLM can answer correctly. Two standard metrics:
- **recall@k** — is the correct chunk in the top-k?
- **MRR** (mean reciprocal rank) — `1/rank` of the correct chunk, averaged. Rewards ranking it
  *higher*, not just *present*.

In [ ]:
# --- YOU WRITE: recall@k and reciprocal rank ---
def retrieved_ids(query, retrieve_fn):
    return [d.metadata["chunk_id"] for d in retrieve_fn(query)]

def recall_at_k(source_id, ids, k):
    return 1.0 if source_id in ids[:k] else 0.0

def reciprocal_rank(source_id, ids):
    return 1.0 / (ids.index(source_id) + 1) if source_id in ids else 0.0

# sanity check on the first gold item
ids = retrieved_ids(gold[0]["question"], retrieve_baseline)
print("recall@3:", recall_at_k(gold[0]["source_id"], ids, 3),
      "| RR:", round(reciprocal_rank(gold[0]["source_id"], ids), 3))

In [ ]:
# --- INSTRUMENT: score the baseline retriever over the whole gold set ---
rows = [(g["source_id"], retrieved_ids(g["question"], retrieve_baseline)) for g in gold]
for k in (1, 3, 5):
    r = np.mean([recall_at_k(s, ids, k) for s, ids in rows])
    print(f"recall@{k}: {r:.2f}")
print(f"MRR:       {np.mean([reciprocal_rank(s, ids) for s, ids in rows]):.3f}")

**🎤 Talk-track — Retrieval metrics.** *"recall@k tells me whether the answer is even
reachable; MRR tells me whether it's ranked where the LLM will actually use it. This is where I'd
point a dashboard first — retrieval regressions are the cheapest bugs to catch and the most
expensive to miss."*

---
## 3 · Answer evaluation — faithfulness & relevance *(Claude as judge)*

Good retrieval doesn't guarantee a good answer — the LLM can still hallucinate or wander. We score
each generated answer with **Claude as an LLM-judge**:
- **Faithfulness** — is every claim supported by the retrieved context? (hallucination check)
- **Relevance** — does the answer actually address the question?

This is where Mistral loads. **Run the load cell once.** If the T4 OOMs, set `USE_LOCAL_LLM = False`
below to generate with Claude instead (note: that changes *what* you're evaluating).

In [ ]:
# --- PROVIDED: the generator (local Mistral, or Claude fallback) ---
USE_LOCAL_LLM = True   # set False if the T4 OOMs

RAG_TEMPLATE = (" [INST] Answer using ONLY the context. If it's not there, say you don't know.\n\n"
                "Context:\n{context}\n\nQuestion: {question} [/INST] Assistant: ")

if USE_LOCAL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
    _mid = "mistralai/Mistral-7B-Instruct-v0.3"
    _qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    print("Loading Mistral (run once) ...")
    _tok = AutoTokenizer.from_pretrained(_mid)
    _model = AutoModelForCausalLM.from_pretrained(_mid, quantization_config=_qc, device_map="auto")
    _pipe = pipeline("text-generation", model=_model, tokenizer=_tok, max_new_tokens=384,
                     temperature=0.3, do_sample=True, repetition_penalty=1.2, return_full_text=False)
    _pipe.model.config.pad_token_id = _pipe.model.config.eos_token_id
    def _gen(prompt): return _pipe(prompt)[0]["generated_text"].strip()
else:
    def _gen(prompt):
        return client.messages.create(model=JUDGE_MODEL, max_tokens=384,
            messages=[{"role": "user", "content": prompt}]).content[0].text.strip()

def generate_answer(question):
    docs = retrieve_reranked(question, top_k=4)
    context = format_docs(docs)
    return _gen(RAG_TEMPLATE.format(context=context, question=question)), context

print("Generator ready. USE_LOCAL_LLM =", USE_LOCAL_LLM)

In [ ]:
# --- PROVIDED: the structured shape Claude must return ---
class Judgment(BaseModel):
    faithfulness: int      # 1 (unsupported) .. 5 (fully supported by context)
    answer_relevance: int  # 1 (off-topic)   .. 5 (directly answers)
    rationale: str

print("Judgment schema ready.")

In [ ]:
# --- YOU WRITE: the Claude LLM-judge ---
def judge_answer(question, context, answer):
    prompt = (
        "You are grading a RAG answer. Score 1-5 each.\n"
        "faithfulness: is EVERY claim in the answer supported by the context?\n"
        "answer_relevance: does the answer address the question?\n\n"
        f"Question: {question}\n\nContext:\n{context}\n\nAnswer:\n{answer}"
    )
    resp = client.messages.parse(
        model=JUDGE_MODEL, max_tokens=512,
        messages=[{"role": "user", "content": prompt}],
        output_format=Judgment)
    return resp.parsed_output

_a, _ctx = generate_answer(gold[0]["question"])
print(judge_answer(gold[0]["question"], _ctx, _a))

In [ ]:
# --- INSTRUMENT: judge the pipeline over a small sample ---
SAMPLE = gold[:8]
results = []
for g in SAMPLE:
    ans, ctx = generate_answer(g["question"])
    j = judge_answer(g["question"], ctx, ans)
    results.append({"q": g["question"], "answer": ans, "j": j})

faith = np.mean([r["j"].faithfulness for r in results])
rel   = np.mean([r["j"].answer_relevance for r in results])
print(f"mean faithfulness: {faith:.2f} / 5")
print(f"mean relevance:    {rel:.2f} / 5\n")
ex = results[0]
print("Example —", ex["q"])
print("Answer:", ex["answer"][:300])
print("Judge:", ex["j"].faithfulness, "/", ex["j"].answer_relevance, "—", ex["j"].rationale)

**🎤 Talk-track — LLM-judge.** *"Faithfulness is my hallucination metric: does the answer
stay inside the retrieved evidence? I use a stronger model as the judge and force structured output
so the scores are machine-readable, not prose I have to parse. In a regulated shop this is the
artifact that lets me say 'here's the measured grounding rate,' not 'it looked fine.'"*

---
## 4 · One controlled experiment — does reranking help?

Change **one knob** — reranking on vs. off — hold everything else fixed, and *measure* the effect
on the same gold set. Retrieval-only, so no Mistral and no OOM risk. This is the difference between
*reranking felt better* and *reranking lifted MRR by X, 95% CI [a, b]*.

In [ ]:
# --- YOU WRITE: evaluate a retrieval function over the gold set ---
def evaluate_retrieval(retrieve_fn):
    recalls = {k: [] for k in (1, 3, 5)}
    rrs = []
    for g in gold:
        ids = retrieved_ids(g["question"], retrieve_fn)
        for k in recalls:
            recalls[k].append(recall_at_k(g["source_id"], ids, k))
        rrs.append(reciprocal_rank(g["source_id"], ids))
    return {**{f"recall@{k}": np.mean(v) for k, v in recalls.items()},
            "MRR": np.mean(rrs), "_rr": rrs}

base = evaluate_retrieval(retrieve_baseline)
rerank = evaluate_retrieval(lambda q: retrieve_reranked(q, top_k=10))

In [ ]:
# --- INSTRUMENT: comparison table + a bootstrap CI on the MRR lift ---
print(f"{'metric':<10}{'baseline':>10}{'rerank':>10}{'delta':>10}")
for m in ("recall@1", "recall@3", "recall@5", "MRR"):
    print(f"{m:<10}{base[m]:>10.3f}{rerank[m]:>10.3f}{rerank[m]-base[m]:>+10.3f}")

# paired bootstrap 95% CI on the per-question MRR difference (the rigor nod)
diffs = np.array(rerank["_rr"]) - np.array(base["_rr"])
rng = np.random.default_rng(0)
boot = [rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"\nMRR lift from reranking: {diffs.mean():+.3f}  (95% CI [{lo:+.3f}, {hi:+.3f}])")
print("CI excludes 0 → the lift is unlikely to be noise." if lo > 0 or hi < 0
      else "CI includes 0 → with this small gold set, the lift isn't distinguishable from noise.")

**🎤 Talk-track — The experiment.** *"This is the loop closing. I isolate one change,
hold everything else fixed, and report an effect size with a confidence interval — not a vibe. The
bootstrap CI is a small version of the causal-inference discipline I bring: I care whether a lift
is real or noise before I ship it. Scale the gold set up and this is a real offline A/B."*

---
## Recap — the feedback loop, closed

| Stage | What you measured | Why it matters in the room |
|-------|-------------------|-----------------------------|
| **Gold set** | synthetic Q → known source chunk | ground truth without hand-labeling |
| **Retrieval** | recall@k, MRR | the ceiling on the whole system; cheapest bugs to catch |
| **Answer** | faithfulness, relevance (Claude judge) | hallucination control; a *measured* grounding rate |
| **Experiment** | rerank on/off, effect size + CI | measure, don't eyeball; real vs. noise |

You now have the full diagram from the prep doc — ingestion → embedding → vector store → retrieval
→ LLM → **evaluation** — with the last stage instrumented. That last stage is the differentiator.

**What production adds next** (the monitoring half): log every query's retrieval scores and judge
scores online; watch for **drift** (embedding distribution shift, faithfulness dropping over time);
alert on regressions; add guardrails (PII filters, refuse-on-no-context). Same metrics, running
continuously instead of once.